# 03 · Model Training

End-to-end training of the GATEncoder + fusion MLP model.

**What this notebook covers:**
- Preparing DataLoaders with the GraphCollator
- Initialising the TCRPeptideBindingModel
- Training with early stopping and checkpointing
- Live loss / AUC curves

## 3.1 Setup

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split

from src.data     import create_sample_data, TCRPeptideDataset, GraphCollator
from src.embedder import ProteinEmbedder
from src.model    import TCRPeptideBindingModel
from src.train    import TCRBindingTrainer
from src.evaluate import plot_training_curves

# ---- Configuration ----
SEED       = 42
DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 8        # increase to 32-64 on GPU
NUM_EPOCHS = 20       # increase to 50-100 for full training
LR         = 1e-4
MODEL_NAME = "facebook/esm2_t12_35M_UR50D"   # swap for 650M in production

torch.manual_seed(SEED)
np.random.seed(SEED)
print(f"Device     : {DEVICE}")
print(f"Batch size : {BATCH_SIZE}")
print(f"Epochs     : {NUM_EPOCHS}")

## 3.2 Prepare data

In [ ]:
# Create synthetic dataset (swap for real VDJdb data in production)
data = create_sample_data(n_samples=500, seed=SEED)

train_data, temp   = train_test_split(data, test_size=0.3,
                                       random_state=SEED, stratify=data["label"])
val_data, test_data = train_test_split(temp, test_size=0.5,
                                        random_state=SEED, stratify=temp["label"])

print(f"Train: {len(train_data)}  Val: {len(val_data)}  Test: {len(test_data)}")
print(f"Label balance (train): {train_data['label'].mean():.2%} positive")

## 3.3 Initialise embedder and DataLoaders

In [ ]:
embedder  = ProteinEmbedder(model_name=MODEL_NAME, device=DEVICE)
collator  = GraphCollator(embedder)

train_loader = DataLoader(TCRPeptideDataset(train_data), batch_size=BATCH_SIZE,
                          shuffle=True,  collate_fn=collator)
val_loader   = DataLoader(TCRPeptideDataset(val_data),   batch_size=BATCH_SIZE,
                          shuffle=False, collate_fn=collator)
test_loader  = DataLoader(TCRPeptideDataset(test_data),  batch_size=BATCH_SIZE,
                          shuffle=False, collate_fn=collator)

print(f"Batches per epoch (train): {len(train_loader)}")

## 3.4 Initialise model

In [ ]:
model = TCRPeptideBindingModel(
    input_dim    = embedder.embed_dim,
    hidden_dim   = 128,
    num_gat_layers = 3,
)

n_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters : {n_params:,}")
print(model)

## 3.5 Train

In [ ]:
trainer = TCRBindingTrainer(
    model            = model,
    device           = DEVICE,
    lr               = LR,
    checkpoint_path  = "../results/best_tcr_model.pt",
    patience         = 8,
)

best_auc = trainer.train(train_loader, val_loader, num_epochs=NUM_EPOCHS)
print(f"\nBest validation AUC: {best_auc:.4f}")

## 3.6 Training curves

In [ ]:
fig = plot_training_curves(
    trainer.history,
    save_path="../results/figures/training_curves.png",
)
plt.show()
print("Saved → results/figures/training_curves.png")

## 3.7 Save training history

In [ ]:
import json, pathlib
pathlib.Path("../results").mkdir(exist_ok=True)
with open("../results/training_history.json", "w") as f:
    json.dump(trainer.history, f, indent=2)
print("History saved → results/training_history.json")
print("\nProceed to notebook 04 for evaluation ►")